# 🦙 Local Text-to-SQL RAG (Ollama & Qwen 2.5)

This notebook demonstrates how to query massive tabular datasets instantly using **DuckDB** and **LangChain**.

We are using **Ollama** to run the AI completely offline on your personal computer! This means **ZERO** rate limits, infinite requests, and completely private data processing.

### Step 1: Install Dependencies

In [2]:
# Install the necessary libraries for our pipeline
#!pip install -q duckdb duckdb-engine langchain langchain-classic langchain-community langchain-ollama sqlalchemy==2.0.44

### Step 2: Build the Database instantly with DuckDB
Instead of loading 1.4M rows into memory (RAM) with Pandas which causes crashes, we build a lightning-fast local analytical database.

In [3]:
import duckdb
import os
import pandas as pd

# File paths
db_path = "apple_sales_rag_ollama.db"
csv_path = "../data/processed/cleaned_apple_sales_enriched_realistic.csv"

# Remove old db if exists to prevent overlapping issues during testing
if os.path.exists(db_path):
    try: os.remove(db_path)
    except: pass

print(f"Connecting to DuckDB and loading massive dataset from {csv_path}...")
con = duckdb.connect(db_path)
con.execute(f"CREATE TABLE sales AS SELECT * FROM read_csv_auto('{csv_path}')")
print("\u2705 Successfully loaded 1 Million rows into DuckDB!")

print("\nSchema (What the Local LLM sees):")
display(con.execute("DESCRIBE sales").df())
con.close()

Connecting to DuckDB and loading massive dataset from ../data/processed/cleaned_apple_sales_enriched_realistic.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Successfully loaded 1 Million rows into DuckDB!

Schema (What the Local LLM sees):


,column_name,column_type,null,key,default,extra
0,sale_id,VARCHAR,YES,None,None,None
1,sale_date,DATE,YES,None,None,None
2,store_id,VARCHAR,YES,None,None,None
3,product_id,VARCHAR,YES,None,None,None
4,quantity,BIGINT,YES,None,None,None
5,product_name,VARCHAR,YES,None,None,None
6,launch_date,DATE,YES,None,None,None
7,price,BIGINT,YES,None,None,None
8,store_name,VARCHAR,YES,None,None,None
9,city,VARCHAR,YES,None,None,None


### Step 3: Advanced AI Prompt Engineering
Because local models don't naturally understand the context of your data, we inject a **Custom System Prompt**. 
This acts as the "brain" or instruction manual for the AI, giving it custom logic hooks for the Apple Retail dataset.

In [4]:
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from langchain_ollama import ChatOllama
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain_core.prompts import PromptTemplate

# 1. Connect LangChain to DuckDB
db = SQLDatabase.from_uri(f"duckdb:///{db_path}")

# 2. Initialize Qwen2.5 
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0)

# 3. MLOPS BEST PRACTICE: The Custom Prompt Template
# This injects strict rules natively into the LangChain Query Engine
custom_prompt = PromptTemplate.from_template(
    """You are an elite DuckDB SQL programming assistant answering questions about Apple Retail Sales data.
    Given an input question, first create a syntactically correct DuckDB query to run, then look at the results of the query and return the answer.
    
    Never query for all the columns from a specific table, only ask for the few relevant columns given the question.
    Be careful to not query for columns that do not exist.
    
    IMPORTANT APPLE RETAIL BUSINESS RULES:
    1. If asked about "Sales", "Revenue", or "Income", ALWAYS default to using the 'sales_amount_realistic' column, NOT the base 'sales_amount' column.
    2. If asked about Volume or Item Counts, ALWAYS use 'quantity_realistic'.
    3. Avoid markdown wrapping. Output only the raw, executable SQL string.

    Only use the following tables:
    {table_info}

    Return a maximum of {top_k} results unless otherwise specified.

    Question: {input}"""
)

# 4. Create the Direct SQL Chain (Text -> SQL -> Result) hooked up to the Prompt
write_query = create_sql_query_chain(llm, db, prompt=custom_prompt)
execute_query = QuerySQLDataBaseTool(db=db)

# 5. Link the writer tool to the executor tool
chain = write_query | execute_query

print("\u2705 Robust Offline SQL Chain (with Advanced Prompting) is ready to roll!")

✅ Robust Offline SQL Chain (with Advanced Prompting) is ready to roll!


d:\anaconda\envs\Apple\Lib\site-packages\duckdb_engine\__init__.py:184: DuckDBEngineWarning: duckdb-engine doesn't yet support reflection on indices
  warnings.warn(
C:\Users\GM\AppData\Local\Temp\ipykernel_1604\2531606138.py:37: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  execute_query = QuerySQLDataBaseTool(db=db)


### Step 4: The 100% Offline SQL Gauntlet!
Let's test both simple questions and extremely hard Data Science database queries.

In [5]:
def ask_local_ai(question):
    print(f"\nQuestion: {question}")
    print("Thinking...")
    
    sql_query = write_query.invoke({"question": question})
    clean_sql = sql_query.replace("```sql", "").replace("```", "").replace("SQLQuery:", "").strip()
    print(f"\u2699 Generated SQL: {clean_sql}\n")
    
    try:
        result = execute_query.invoke(clean_sql)
        print(f"====== FINAL ANSWER ======\n{result}\n")
    except Exception as e:
        print(f"Error running SQL: {e}\n")

#### Easy Level Tests (Standard Analytics)

In [6]:
ask_local_ai("How many unique stores do we have in our entire dataset?")
ask_local_ai("What are the distinct product categories we sell? List them out.")
ask_local_ai("Which country had the highest number of overall sales transactions (not volume, just number of rows)?")


Question: How many unique stores do we have in our entire dataset?
Thinking...
⚙ Generated SQL: SELECT COUNT(DISTINCT store_id) AS unique_store_count FROM sales;

====== FINAL ANSWER ======
[(75,)]


Question: What are the distinct product categories we sell? List them out.
Thinking...
⚙ Generated SQL: SELECT DISTINCT category_name FROM sales;

====== FINAL ANSWER ======
[('Accessories',), ('Smartphone',), ('Tablet',), ('Streaming Device',), ('Wearable',), ('Audio',), ('Subscription Service',), ('Laptop',), ('Smart Speaker',), ('Desktop',)]


Question: Which country had the highest number of overall sales transactions (not volume, just number of rows)?
Thinking...
⚙ Generated SQL: SELECT city, COUNT(*) AS transaction_count
FROM sales
GROUP BY city
ORDER BY transaction_count DESC
LIMIT 1;

====== FINAL ANSWER ======
[('Dubai', 55619)]



#### Medium Level Tests (Mathematical Inference & Data rules)

In [7]:
ask_local_ai("Which country sold the absolute most physical items (volume) out of all the countries combined?")
ask_local_ai("What is the average Apple revenue for the iPhone 14 in Japan in 2024? Remember to use the realistic amount.")
ask_local_ai("Are there any products where the quantity sold was high but the sales generated was zero?")


Question: Which country sold the absolute most physical items (volume) out of all the countries combined?
Thinking...
⚙ Generated SQL: SELECT city, SUM(quantity_realistic) AS total_quantity
FROM sales
GROUP BY city
ORDER BY total_quantity DESC
LIMIT 1;

====== FINAL ANSWER ======
[('London', 352330)]


Question: What is the average Apple revenue for the iPhone 14 in Japan in 2024? Remember to use the realistic amount.
Thinking...
⚙ Generated SQL: SELECT AVG(sales_amount_realistic) AS average_revenue
FROM sales
WHERE product_name = 'iPhone 14' AND city = 'japan' AND year = 2024;

====== FINAL ANSWER ======
[(None,)]


Question: Are there any products where the quantity sold was high but the sales generated was zero?
Thinking...
⚙ Generated SQL: SELECT product_name, quantity_realistic, sales_amount_realistic
FROM sales
WHERE quantity_realistic > 10 AND sales_amount_realistic = 0;

====== FINAL ANSWER ======




#### Advanced Level Tests (HAVING clauses & Time Series Grouping)

In [8]:
ask_local_ai("Show me the top 3 stores with the highest average promo_flag impact, but filter out any store with less than 1000 total sales transactions.")
ask_local_ai("What was the total revenue generated for MacBooks in the United States grouped by month in 2024? Sort it from January to December.")


Question: Show me the top 3 stores with the highest average promo_flag impact, but filter out any store with less than 1000 total sales transactions.
Thinking...
⚙ Generated SQL: SELECT store_name, AVG(promo_flag) AS avg_promo_flag_impact
FROM sales
GROUP BY store_name
HAVING SUM(quantity_realistic) >= 1000
ORDER BY avg_promo_flag_impact DESC
LIMIT 3;

====== FINAL ANSWER ======
[('Apple Kumamoto', 0.10470605063653268), ('Apple Piazza Liberty', 0.10467248587985037), ('Apple Opera', 0.10458596894767108)]


Question: What was the total revenue generated for MacBooks in the United States grouped by month in 2024? Sort it from January to December.
Thinking...
⚙ Generated SQL: SELECT 
    T1.month,
    SUM(T1.sales_amount_realistic) AS total_revenue
FROM 
    sales AS T1
JOIN 
    categories AS T2 ON T1.category_id = T2.category_id
WHERE 
    T2.category_name = 'MacBook' AND 
    T1.country_norm_mapped = 'united states' AND 
    T1.year = 2024
GROUP BY 
    T1.month
ORDER BY 
    T1.month;

In [9]:
ask_local_ai("top 5 stores")


Question: top 5 stores
Thinking...
⚙ Generated SQL: SELECT store_name, SUM(sales_amount_realistic) AS total_sales
FROM sales
GROUP BY store_name
ORDER BY total_sales DESC
LIMIT 5;

====== FINAL ANSWER ======
[('Apple Orchard Road', 207723826.5543254), ('Apple Covent Garden', 194616425.7047786), ('Apple Chadstone', 175701542.7835955), ('Apple Southland', 170849759.58883607), ('Apple Ala Moana', 166836488.1611117)]



In [10]:
ask_local_ai("top 10 products")


Question: top 10 products
Thinking...
⚙ Generated SQL: SELECT product_name, SUM(sales_amount_realistic) AS total_sales
FROM sales
GROUP BY product_name
ORDER BY total_sales DESC
LIMIT 10;

====== FINAL ANSWER ======
[('iPad (9th Generation)', 177403720.75375724), ('Apple Music', 172294182.7436231), ('MagSafe Charger', 167163510.0897399), ('AirPods Max', 161409256.72018072), ('Beats Solo Pro', 160873133.3823157), ('Apple Watch Hermès', 147912154.47718558), ('iPhone 12 mini', 145920634.82393503), ('AirPods (3rd Generation)', 143764152.2275807), ('iPad (10th Generation)', 141222623.32397258), ('Apple Watch Series 7', 137493787.3614074)]



In [11]:
ask_local_ai("top city")


Question: top city
Thinking...
⚙ Generated SQL: SELECT city, SUM(sales_amount_realistic) AS total_sales
FROM sales
GROUP BY city
ORDER BY total_sales DESC
LIMIT 5;

====== FINAL ANSWER ======
[('London', 378775121.9254321), ('New York', 375926150.1870714), ('Paris', 335460549.95310044), ('Singapore', 326911232.4200075), ('Dubai', 319978135.93912065)]



In [12]:
ask_local_ai("how many columns do we have not from sales")


Question: how many columns do we have not from sales
Thinking...
⚙ Generated SQL: SELECT count(*) FROM information_schema.columns WHERE table_name = 'sales' EXCEPT SELECT count(*) FROM information_schema.columns WHERE table_name = 'sales';

====== FINAL ANSWER ======




In [13]:
ask_local_ai("how many row do we have")


Question: how many row do we have
Thinking...
⚙ Generated SQL: SELECT COUNT(*) FROM sales;

====== FINAL ANSWER ======
[(1040200,)]



In [20]:
ask_local_ai("what is 10 products by price with out duplicate ")


Question: what is 10 products by price with out duplicate 
Thinking...
⚙ Generated SQL: SELECT product_name, price_realistic
FROM sales
GROUP BY product_name, price_realistic
ORDER BY price_realistic DESC
LIMIT 10;

====== FINAL ANSWER ======
[('Apple Music', 2427.0584165302157), ('Apple Music', 2362.290441070545), ('iPad mini (5th Generation)', 2356.7527067907467), ('Apple Music', 2338.401352664711), ('Apple Music', 2331.557626923838), ('iPad (9th Generation)', 2330.9002965710156), ('Apple Music', 2322.5587094981365), ('Apple Music', 2322.2502644315773), ('Apple Music', 2322.204628382021), ('Apple Music', 2320.4232511926)]



In [17]:
ask_local_ai ("what is gdp of london by year")


Question: what is gdp of london by year
Thinking...
⚙ Generated SQL: SELECT DISTINCT year, SUM(gdp_per_capita) AS total_gdp
FROM sales
WHERE city = 'London'
GROUP BY year;

====== FINAL ANSWER ======
[(2023, 560200424.3679081), (2020, 463241103.3122627), (2024, 523420205.8509979), (2022, 520143299.00170124), (2021, 536838882.46555305)]



In [22]:
ask_local_ai("i phone 13 and 14 price by year ")


Question: i phone 13 and 14 price by year 
Thinking...
⚙ Generated SQL: SELECT product_name, price_realistic, year
FROM sales
WHERE product_name IN ('iPhone 13', 'iPhone 14')
ORDER BY year;

====== FINAL ANSWER ======
[('iPhone 13', 612.851370377835, 2020), ('iPhone 13', 648.3590064012783, 2020), ('iPhone 13', 646.1072441345137, 2020), ('iPhone 13', 642.9380491880298, 2020), ('iPhone 13', 608.2187688936616, 2020), ('iPhone 13', 619.6663929026249, 2020), ('iPhone 14', 1462.8217754014543, 2020), ('iPhone 13', 616.5184404223688, 2020), ('iPhone 14', 1586.449162260585, 2020), ('iPhone 14', 1457.485930043882, 2020), ('iPhone 14', 1507.6953113770621, 2020), ('iPhone 14', 1456.8630100725723, 2020), ('iPhone 13', 660.7239118923599, 2020), ('iPhone 13', 638.5577090978531, 2020), ('iPhone 14', 1571.0380684813717, 2020), ('iPhone 14', 1423.7753002887716, 2020), ('iPhone 14', 1424.0440130607844, 2020), ('iPhone 13', 619.5686937527847, 2020), ('iPhone 14', 1460.931322339955, 2020), ('iPhone 14', 1

In [26]:
ask_local_ai("how many dollar to buy one  i phone 14 ")


Question: how many dollar to buy one  i phone 14 
Thinking...
⚙ Generated SQL: SELECT SUM(sales_amount_realistic) AS total_sales FROM sales WHERE product_name = 'iPhone 14';

====== FINAL ANSWER ======
[(62303665.26745336,)]



In [28]:
ask_local_ai("what if i have 2000 dollar this is the price of i phone 14")


Question: what if i have 2000 dollar this is the price of i phone 14
Thinking...
⚙ Generated SQL: SELECT product_name, sales_amount_realistic 
FROM sales 
WHERE price_realistic = 2000;

====== FINAL ANSWER ======


